# MLflow 2: the Model Registry

Tracking answers *"what did we try?"*. The registry answers *"which model is in use,
who approved it, and what was it trained on?"*

| Registry concept | In this lesson |
|---|---|
| **Registered model** | `gam-risk`: the name the clinic's software asks for |
| **Version** | `gam-risk` v1, v2, ...: each points at one logged model from one run |
| **Alias** | `@champion` = the version in use. Software loads `models:/gam-risk@champion` and never hard-codes a version number |
| **Tags / description** | `validation_status=approved`, who approved it, why |

(Older MLflow used *stages* like "Staging"/"Production"; they're deprecated in favour of aliases.)

Run `1_tracking.ipynb` first, and keep `mlflow ui --port 5000` running.

In [ ]:
import os
import warnings

os.environ["MLFLOW_DISABLE_AGENT_HINT"] = "1"      # silence an MLflow tip about LLM tracing
os.environ["MLFLOW_DISABLE_TELEMETRY"] = "true"   # don't send anonymous usage data to MLflow
warnings.filterwarnings("ignore", category=UserWarning)

import mlflow
import pandas as pd

import emr_model as em                              # this lesson's data/model/evaluation helpers
from train import EXPERIMENT, train_and_log        # one function = one fully logged run

pd.set_option("display.width", 160, "display.max_columns", 20)
print("MLflow", mlflow.__version__, "| tracking URI:", mlflow.get_tracking_uri())

In [ ]:
from mlflow import MlflowClient

client = MlflowClient()
MODEL = "gam-risk"


def model_uri_for(run_id):
    # The model logged by a run (MLflow 3: models are objects of their own, linked to runs)
    found = mlflow.search_logged_models(filter_string=f"source_run_id = '{run_id}'",
                                        experiment_ids=[mlflow.get_experiment_by_name(EXPERIMENT).experiment_id],
                                        output_format="list")
    return found[0].model_uri
_ = mlflow.set_experiment(EXPERIMENT)

## 1. Register the best run as version 1

Pick the run to register: best ROC-AUC among the runs that used the 2024 train/test
months (excluding the autolog demo, which has no test metrics).

In [ ]:
candidates = mlflow.search_runs(
    experiment_names=[EXPERIMENT],
    filter_string="params.test_until = '2024-12' and metrics.roc_auc > 0",
    order_by=["metrics.roc_auc DESC", "attributes.start_time ASC"],
)
best = candidates.iloc[0]
print("registering run", best["tags.mlflow.runName"], "roc_auc", round(best["metrics.roc_auc"], 4))

v1 = mlflow.register_model(model_uri_for(best.run_id), MODEL)       # creates gam-risk, version 1
client.update_model_version(MODEL, v1.version,
                            description="Logistic regression, trained Jan-Oct 2024.")
client.set_model_version_tag(MODEL, v1.version, "validation_status", "approved")
client.set_model_version_tag(MODEL, v1.version, "approved_by", "nutrition-ml-team")
client.set_registered_model_alias(MODEL, "champion", v1.version)   # this version is now in use
print("champion is version", client.get_model_version_by_alias(MODEL, "champion").version)

**In the UI:** *Model registry* (left menu) → *gam-risk*: version 1, alias `@champion`,
tags, description, and a link back to the run (params, data_md5, git_commit). That link
is the audit trail.

## 2. Use the champion

Whoever scores new visits (a batch job, the EMR's backend) loads the model **by alias**.
When the champion changes, they get the new one without changing their code.

In [ ]:
em.simulate("2025-01", 6, scenario="concept_shift")   # Jan-Jun 2025 arrive (see section 3)

champion = mlflow.pyfunc.load_model(f"models:/{MODEL}@champion")
january = em.load_months("2025-01", "2025-01")
risk = champion.predict(january[em.FEATURES])[:, 1]    # column 1 = probability of GAM within 90 days

january["risk"] = risk
print(f"{len(january)} visits scored; top 5 by risk:")
january.nlargest(5, "risk")[["visit_id", "facility_id", "age_months", "sex", "muac_cm", "whz", "risk"]].round(3)

### What the signature protects, and what it doesn't

The model enforces its **signature**, the input contract recorded when it was logged.
Try a few broken inputs:

In [ ]:
import numpy as np

x = january[em.FEATURES].head(3)

def attempt(label, df):
    try:
        print(f"{label:40s} accepted -> risk {np.round(champion.predict(df)[:, 1], 3)}")
    except Exception as e:
        print(f"{label:40s} BLOCKED  -> {str(e).split('Error:')[-1].strip()[:70]}")

attempt("normal visits", x)
attempt("missing column (sex)", x.drop(columns="sex"))
attempt("wrong type (age as text)", x.assign(age_months=["14", "20", "3"]))
attempt("unknown code (sex = 'female')", x.assign(sex=["female", "M", "F"]))
attempt("MUAC in mm (137 instead of 13.7)", x.assign(muac_cm=x["muac_cm"] * 10))

The signature checks **shape and type**: missing columns and wrong types are blocked.
It does **not** check **meaning**: an unknown code is silently ignored, and MUAC in
millimetres is accepted and drives every child's risk to about **zero**: the model
confidently says nobody needs follow-up.

So the signature and Great Expectations overlap only a little:

| | Model signature | Great Expectations |
|---|---|---|
| When | every prediction call | each batch, before training or scoring |
| Checks | column names, types, required | ranges, units, codes, completeness, uniqueness, distributions, cohort rules |
| Catches mm vs cm? | **no** | yes |

Use both: the signature is the model's last line of defence; GX is the real data check.

## 3. The world changes: champion vs challenger

In 2025 the simulated region loses general food distribution (`concept_shift`
scenario): illness and long walks to water now predict malnutrition much more
strongly than in 2024. Prevalence jumps. Is the 2024 champion still good enough,
or should a model retrained on recent data replace it?

**The fair way to decide:** evaluate both models on the **same** recent months that
*neither* was trained on: May–Jun 2025. (In the DVC lesson the test set moved with
every version, which made comparisons misleading. A fixed benchmark fixes that.)

In [ ]:
# Challenger: same recipe, retrained with data up to April 2025, logged as a normal run
challenger_run, _, challenger_uri = train_and_log(model="logistic", C=1.0, train_until="2025-04",
                                                  test_from="2025-05", test_until="2025-06",
                                                  run_name="challenger-retrained-2025-04")

benchmark = em.load_months("2025-05", "2025-06")
challenger = mlflow.sklearn.load_model(challenger_uri)
champion_sk = mlflow.sklearn.load_model(f"models:/{MODEL}@champion")

rows = {}
for name, m in [("champion (v1)", champion_sk), ("challenger", challenger)]:
    overall, by_group = em.evaluate(m, benchmark)
    rows[name] = {"roc_auc": overall["roc_auc"], "sensitivity": overall["sensitivity"],
                  "ppv": overall["ppv"],
                  "sens_girls": by_group["sex"]["F"]["sensitivity"],
                  "sens_boys": by_group["sex"]["M"]["sensitivity"]}
comparison = pd.DataFrame(rows).T
print(f"benchmark: {len(benchmark)} visits, prevalence {benchmark[em.LABEL].mean():.1%}")
comparison.round(4)

### A written-down promotion rule

Decide the rule *before* looking at results, write it as code, and log the decision.
Ours: the challenger must improve ROC-AUC by at least 0.005, and must not reduce
sensitivity for girls or for boys by more than 0.02.

In [ ]:
def should_promote(champ, chall, min_auc_gain=0.005, max_group_drop=0.02):
    reasons = []
    if chall["roc_auc"] - champ["roc_auc"] < min_auc_gain:
        reasons.append(f"ROC-AUC gain {chall['roc_auc'] - champ['roc_auc']:+.4f} < {min_auc_gain}")
    for g in ["sens_girls", "sens_boys"]:
        if champ[g] - chall[g] > max_group_drop:
            reasons.append(f"{g} drops by {champ[g] - chall[g]:.3f}")
    return not reasons, reasons

promote, reasons = should_promote(comparison.loc["champion (v1)"], comparison.loc["challenger"])
print("PROMOTE" if promote else "KEEP CHAMPION", reasons)

In [ ]:
if promote:
    v2 = mlflow.register_model(challenger_uri, MODEL)
    client.update_model_version(MODEL, v2.version,
        description="Logistic regression retrained Jan 2024-Apr 2025 after food distribution stopped.")
    client.set_model_version_tag(MODEL, v2.version, "validation_status", "approved")
    client.set_model_version_tag(MODEL, v2.version, "benchmark", "2025-05..2025-06")
    client.set_registered_model_alias(MODEL, "previous_champion", v1.version)   # remember the old one
    client.set_registered_model_alias(MODEL, "champion", v2.version)            # switch
print("champion is now version", client.get_model_version_by_alias(MODEL, "champion").version)

Anything loading `models:/gam-risk@champion` now gets version 2. Nobody changed their code.

**Rollback** is the same one line, pointing the alias back:
```python
client.set_registered_model_alias(MODEL, "champion", "1")
```

**Discussion:** in a *stable* period (no concept shift), the same retraining gave no
real gain (ROC-AUC 0.818 vs 0.819 on the benchmark): newer isn't automatically
better. A fixed benchmark and a written rule are what stop "retrain every month and
ship it" from quietly making things worse.

## 4. The audit trail

*"Which model flagged this child in June, and what was it trained on?"* Follow the
links: alias → version → run → params and tags.

In [ ]:
mv = client.get_model_version_by_alias(MODEL, "champion")
r = mlflow.get_run(mv.run_id)
print(f"gam-risk@champion = version {mv.version}  ({mv.tags.get('validation_status')}, "
      f"benchmark {mv.tags.get('benchmark')})")
print("  run:          ", r.info.run_name, r.info.run_id[:8])
print("  trained on:   ", r.data.params["train_from"], "to", r.data.params["train_until"],
      "| data_md5", r.data.tags["data_md5"])
print("  code:         ", "git", r.data.tags["git_commit"])
print("  settings:     ", {k: r.data.params[k] for k in ["model", "C", "alert_rate"]})

`data_md5` identifies the exact monthly files, playing the role of the hash in DVC's
`data/raw.dvc`. When the pipeline is assembled (Prefect lesson), the DVC hash itself
is logged here, connecting MLflow's model history to DVC's data history.

## 5. Serve the champion over HTTP

In a terminal, from this folder (leave `mlflow ui` running in another one):
```bash
mlflow models serve -m "models:/gam-risk@champion" --port 5001 --env-manager local
```
Then run the next cell: it sends three visits as JSON, like the EMR's backend would.

In [ ]:
import requests

payload = {"dataframe_split": january[em.FEATURES].head(3).to_dict(orient="split")}
try:
    resp = requests.post("http://127.0.0.1:5001/invocations", json=payload, timeout=10)
    print(resp.status_code, resp.json())
except requests.ConnectionError:
    print("Server not running: start `mlflow models serve ...` in a terminal first.")

### What to remember
- The registry separates *"we trained it"* (a run) from *"we use it"* (`@champion`).
- Promotion is a **decision with evidence**: same benchmark for both models, a rule
  written in advance, per-group checks, and a record of who approved it.
- Rollback is moving an alias, not a redeploy.
- Every registered version links back to its run, data and code: the audit trail.